# 📦 GGUF 匯出 — llama32-taiwanchat-qlora

把已經訓練好、已經合併的完整模型轉成 **GGUF** 格式,推上 HuggingFace,供 [Ollama](https://ollama.com)、[LM Studio](https://lmstudio.ai) 等本機推論工具直接下載使用。

這個 notebook**不做任何訓練**,只是載入已經在 HF 上的合併模型、轉檔、上傳,所以比訓練 notebook 快很多(每個模型約 10–20 分鐘)。

**執行三步驟:**
1. Runtime → Change runtime type → **A100**(8B fp16 權重約 16GB,建議用大顯存 GPU)。
2. 左側 🔑 Secrets 確認 `HF_TOKEN`(write 權限)還在、Notebook access 開著。
3. 參數 cell 預設就會匯出 3B 與 8B 兩個模型,不用改就能 **Runtime → Run all**。


In [ ]:
# ============ 參數(只需要改這裡) ============
HF_USERNAME         = ""     # 留空 → 自動用 HF token 對應的帳號
QUANTIZATION_METHOD = "q4_k_m"   # 推薦預設(平衡品質/檔案大小);其他選項見 README 或 Unsloth 文件,例如 "q8_0"(品質較高、檔案較大)、"f16"(不量化,轉檔最快)

# 要匯出的模型:(已合併的 fp16 repo, 匯出後的 GGUF repo 名稱後綴)
MODELS_TO_EXPORT = [
    {"merged_repo": "llama-3.2-3b-taiwan-chat", "gguf_suffix": "-gguf"},
    {"merged_repo": "llama-3.1-8b-taiwan-chat",  "gguf_suffix": "-gguf"},
]
MAX_SEQ_LENGTH = 2048

print(f"將匯出 {len(MODELS_TO_EXPORT)} 個模型,量化方式:{QUANTIZATION_METHOD}")
for m in MODELS_TO_EXPORT:
    print(f"  {m['merged_repo']} → {m['merged_repo']}{m['gguf_suffix']}")


In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2


In [ ]:
# 環境檢查:上面的安裝 cell 有 %%capture(安靜),這裡響亮地驗證一切就緒
import locale
locale.getpreferredencoding = lambda *args: "UTF-8"   # 已知 Colab locale bug 的 workaround

import torch
assert torch.cuda.is_available(), (
    "偵測不到 GPU!請點 Runtime → Change runtime type → 選 A100,再 Runtime → Run all。"
)
import unsloth
import transformers
GPU_NAME = torch.cuda.get_device_name(0)
print(f"GPU = {GPU_NAME}")
print(f"torch={torch.__version__}  transformers={transformers.__version__}  unsloth={unsloth.__version__}")


In [ ]:
# HF 登入
from google.colab import userdata
from huggingface_hub import HfApi, login

HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, (
    "讀不到 HF_TOKEN!請點左側 🔑(Secrets)→ Add new secret:名稱 HF_TOKEN、"
    "值為 HuggingFace write token(hf.co/settings/tokens),打開 Notebook access 後重跑本 cell。"
)
login(token=HF_TOKEN)
api = HfApi(token=HF_TOKEN)
HF_USERNAME = HF_USERNAME or api.whoami()["name"]
print(f"HF_USERNAME = {HF_USERNAME}")


In [ ]:
# 逐一載入合併模型 → 轉 GGUF → 推送。不需要重跑訓練或 LoRA。
import gc
from huggingface_hub import snapshot_download
from unsloth import FastLanguageModel

for m in MODELS_TO_EXPORT:
    merged_repo = f"{HF_USERNAME}/{m['merged_repo']}"
    gguf_repo   = f"{HF_USERNAME}/{m['merged_repo']}{m['gguf_suffix']}"
    print("=" * 80)
    print(f"下載 {merged_repo} …")
    # 先把檔案完整抓進本機快取:unsloth 的載入器若遇到短暫網路問題會自動切換
    # 「離線模式」重試,若此時本機還沒有快取會直接壞掉(已知的錯誤處理漏洞,
    # 會丟出 AttributeError: 'NoneType' object has no attribute 'endswith'）。
    # snapshot_download 自帶重試/續傳,先跑這個能避開該問題。
    snapshot_download(repo_id=merged_repo, token=HF_TOKEN)

    print(f"載入 {merged_repo} …")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = merged_repo,
        max_seq_length = MAX_SEQ_LENGTH,
        dtype = None,
        load_in_4bit = False,   # 已經是 fp16 合併權重,不要在載入時再量化
        token = HF_TOKEN,
    )

    print(f"轉檔並推送 GGUF({QUANTIZATION_METHOD})→ {gguf_repo} …")
    model.push_to_hub_gguf(
        gguf_repo,
        tokenizer,
        quantization_method = QUANTIZATION_METHOD,
        token = HF_TOKEN,
    )
    print(f"✅ {merged_repo} → https://huggingface.co/{gguf_repo}")

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

print("=" * 80)
print("全部匯出完成。")


## ✅ 完成!

- 3B GGUF:`https://huggingface.co/<HF_USERNAME>/llama-3.2-3b-taiwan-chat-gguf`
- 8B GGUF:`https://huggingface.co/<HF_USERNAME>/llama-3.1-8b-taiwan-chat-gguf`

**在 Ollama 使用**(下載 repo 裡的 `.gguf` 檔後):

```bash
ollama create taiwan-chat -f Modelfile   # Modelfile 內容: FROM ./該檔案.gguf
ollama run taiwan-chat
```

或直接用 [LM Studio](https://lmstudio.ai) 搜尋你的 HF 帳號名稱下載使用。
